# Table Data Summary


In [1]:
# === notebook config (auto-managed; edit values, not the tag) ===
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Database
DB_PATH = "../data/humans_clean.duckdb"

# Figure style — minimal, Nature/Science publication standard
FIGSIZE = (8, 5)
DPI = 120
FONT_TITLE = 16
FONT_LABEL = 13
FONT_TICK = 11
FONT_LEGEND = 10

# Light, restrained palette (avoid AI-slop saturation)
COLOR_PRIMARY = "#2171b5"
COLOR_SECONDARY = "#b5542a"
COLOR_NEUTRAL = "#7f7f7f"
COLOR_LIGHT = "#d9d9d9"
COLOR_ACCENT = "#6a9e3a"
PALETTE = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT, COLOR_NEUTRAL, COLOR_LIGHT]

import matplotlib as _mpl
_mpl.rcParams.update({
    "figure.figsize": FIGSIZE,
    "figure.dpi": DPI,
    "axes.titlesize": FONT_TITLE,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})


## Data Summary: Field Coverage

This notebook summarizes the availability of key fields (birthdate, deathdate, birthplace, nationality, etc.) across all individuals in the database.
Each row shows how many individuals have a non-null value for that property, along with the percentage of the total population.

In [2]:
import duckdb
import pandas as pd

DB = DB_PATH
conn = duckdb.connect(DB, read_only=True)

queries = [
    ('Total individuals',            'SELECT COUNT(*) FROM individuals'),
    ('Date of birth',                'SELECT COUNT(*) FROM individuals WHERE birthdate IS NOT NULL'),
    ('Date of death',                'SELECT COUNT(*) FROM individuals WHERE deathdate IS NOT NULL'),
    ('Place of birth',               'SELECT COUNT(*) FROM individuals WHERE birthcity_en IS NOT NULL'),
    ('Place of death',               'SELECT COUNT(*) FROM individuals WHERE deathcity_en IS NOT NULL'),
    ('Nationality',                  'SELECT COUNT(*) FROM individuals WHERE country_of_citizenship_en IS NOT NULL'),
    ('Occupation',                   'SELECT COUNT(*) FROM individuals WHERE occupations_en IS NOT NULL'),
    ('Gender',                       'SELECT COUNT(*) FROM individuals WHERE gender IS NOT NULL'),
    ('Writing language',             'SELECT COUNT(*) FROM individuals WHERE writing_language_name_en IS NOT NULL'),
    ('Wikipedia wikimedia_links (>=1)',    'SELECT COUNT(*) FROM individuals WHERE wikimedia_links_count > 0'),
    ('External identifiers (>=1)',   'SELECT COUNT(*) FROM individuals WHERE identifiers_count > 0'),
    ('Impact date (computed)',       'SELECT COUNT(*) FROM individuals_floruit_period'),
    ('Cliopatria polity mapping',    'SELECT COUNT(DISTINCT wikidata_id) FROM individuals_cliopatria'),
]

total = None
rows_data = []
for label, q in queries:
    count = conn.execute(q).fetchone()[0]
    if total is None:
        total = count
    rows_data.append({'Property': label, 'Individuals': f'{count:,}', '% of total': f'{100*count/total:.1f}'})

conn.close()

df = pd.DataFrame(rows_data)
df.style.hide(axis='index')


Property,Individuals,% of total
Total individuals,"13,003,420",100.0
Date of birth,"7,511,504",57.8
Date of death,"3,573,591",27.5
Place of birth,"3,724,907",28.6
Place of death,"1,585,590",12.2
Nationality,"5,570,202",42.8
Occupation,"9,032,397",69.5
Gender,"8,954,240",68.9
Writing language,"225,836",1.7
Wikipedia wikimedia_links (>=1),"4,960,082",38.1
